In [1]:
# Step 1: Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor

from sklearn.metrics import mean_squared_error, r2_score


# Step 2: Load Dataset

df = pd.read_csv("housing.csv")

# Clean column names
df.columns = df.columns.str.strip()

# Rename target column
df.rename(columns={"median_house_value": "HousePrice"}, inplace=True)

# Drop categorical column
df = df.drop("ocean_proximity", axis=1)

# 🔥 👉 PUT IT HERE (THIS IS THE CORRECT PLACE)
df = df.fillna(df.mean())

# Step 3: Separate features and target
X = df.drop("HousePrice", axis=1)
y = df["HousePrice"]

# Step 3: Separate Features and Target
X = df.drop("HousePrice", axis=1)
y = df["HousePrice"]


# Step 4: Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# Step 5: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)


# Step 6: Overfitting Detection (Decision Tree)
tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)

train_pred = tree.predict(X_train)
test_pred = tree.predict(X_test)

train_rmse = mean_squared_error(y_train, train_pred, squared=False)
test_rmse = mean_squared_error(y_test, test_pred, squared=False)

print("Train RMSE:", train_rmse)
print("Test RMSE:", test_rmse)


# Step 7: Cross-Validation
cv_scores = cross_val_score(
    tree, X_scaled, y,
    scoring="neg_root_mean_squared_error",
    cv=5
)
cv_rmse = -cv_scores.mean()
print("Cross-Validation RMSE:", cv_rmse)


# Step 8: Hyperparameter Tuning (Decision Tree)
param_grid = {
    "max_depth": [3, 5, 7, 10],
    "min_samples_split": [2, 5, 10]
}

grid = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)


# Step 9: Evaluate Optimized Model
best_tree = grid.best_estimator_

y_pred_tree = best_tree.predict(X_test)

tree_rmse = mean_squared_error(y_test, y_pred_tree, squared=False)
tree_r2 = r2_score(y_test, y_pred_tree)

print("Tuned Decision Tree RMSE:", tree_rmse)
print("Tuned Decision Tree R2:", tree_r2)


# Step 10: Baseline Models (for comparison)

# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
lr_rmse = mean_squared_error(y_test, y_pred_lr, squared=False)
lr_r2 = r2_score(y_test, y_pred_lr)

# Ridge Regression
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)
ridge_rmse = mean_squared_error(y_test, y_pred_ridge, squared=False)
ridge_r2 = r2_score(y_test, y_pred_ridge)


# Step 11: Comparison Table
results = pd.DataFrame({
    "Model": ["Linear Regression", "Ridge Regression", "Tuned Decision Tree"],
    "RMSE": [lr_rmse, ridge_rmse, tree_rmse],
    "R2 Score": [lr_r2, ridge_r2, tree_r2]
})

print(results)

import matplotlib.pyplot as plt

models = ["Linear Regression", "Ridge", "Tuned Decision Tree"]
rmse_values = [lr_rmse, ridge_rmse, tree_rmse]

plt.figure()
plt.bar(models, rmse_values)
plt.title("Model Comparison (RMSE)")
plt.xlabel("Models")
plt.ylabel("RMSE")
plt.xticks(rotation=20)

for i, v in enumerate(rmse_values):
    plt.text(i, v, f"{v:.0f}", ha='center', va='bottom')

plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'housing.csv'